# CNN Fundamentals using PyTorch

### Import and verify

In [ ]:
import torch

print(torch.__version__)

2.11.0+cpu


### Image as Tensors

**1. The Digital Grid (Spatial Dimensions)**
*   **Concept:** A computer does not "see" shapes or lighting; it only reads discrete numerical values arranged in a rigid 2D grid. Each discrete point in this grid is a pixel (picture element).
*   **Mathematics:** If an image is $H$ pixels high and $W$ pixels wide, the spatial resolution is mathematically represented as a matrix $M \in \mathbb{R}^{H \times W}$.

**2. The Depth Dimension (Color Channels)**
*   **Concept:** To represent color, the computer stacks multiple spatial matrices on top of each other. These are called **Channels**.
    *   A grayscale image has 1 channel (representing intensity from black to white).
    *   A standard digital color image uses the RGB color model, meaning it is composed of 3 distinct channels: Red, Green, and Blue.
*   **Mathematics:** The complete color image is a Rank-3 Tensor $T \in \mathbb{R}^{C \times H \times W}$. A $256 \times 256$ pixel RGB image is mathematically a tensor of shape `(3, 256, 256)`.

**3. Pixel Value Storage (Data Types)**
*   **Concept:** In standard file formats (like JPEG or PNG), the value of a single color channel for a single pixel is stored as an 8-bit unsigned integer (`uint8`).
*   **Mathematics:** An 8-bit integer can hold $2^8 = 256$ discrete values. Therefore, the numerical domain of raw image data is strictly bounded: $x \in [0, 255]$.
    *   $0$ represents the absolute absence of that color (black).
    *   $255$ represents the maximum intensity of that color.

**4. The Mathematics of Image Normalization**
*   **The CS Problem:** Standard neural networks rely on gradient descent. If we feed a matrix containing values up to $255$ into a network initialized with tiny weights (e.g., $0.01$), the dot products will produce massive numbers. This causes gradients to destabilize (explode) during the backward pass, mathematically preventing the network from converging.
*   **The Solution (Min-Max Scaling):** We must compress the $0-255$ integer range into a floating-point range of $0.0$ to $1.0$.
    *   **Formula:** $x_{scaled} = \frac{x_{raw}}{255.0}$
*   **The Solution (Standardization/Z-Score Normalization):** To further optimize hardware computation, computer scientists go a step further. They calculate the statistical Mean ($\mu$) and Standard Deviation ($\sigma$) of the entire image dataset, and shift the tensor values so the data is centered around zero with a standard deviation of 1.
    *   **Formula:** $x_{norm} = \frac{x_{scaled} - \mu}{\sigma}$

**5. The GPU Batch Format (B, C, H, W)**
*   **Concept:** As we learned in the previous module, GPUs are designed for massive parallel throughput. We never send a single `(C, H, W)` image to the GPU. We stack $B$ number of images into a contiguous block of memory.
*   **Mathematics:** The final Rank-4 Tensor sent to the hardware is $T_{batch} \in \mathbb{R}^{B \times C \times H \times W}$.

In [ ]:
""" IMAGE LOADING, TRANSFORMATION AND EXECUTION PIPELINE """

import torch
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

print(torch.__version__)
print(np.__version__)
print(Image.__version__)

2.11.0+cu128
2.0.2
11.3.0


In [ ]:
# Set the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Hardware: {device}")

Target Hardware: cuda


In [ ]:
# Simulating reading a raw image (.jpg) from SSD / HDD

# Generating a random 256x256 (H x W) image with random pixel values between 0 - 255
# Current shape in Height x Width x Channels and the type is 8 bit unsigned integer
raw_memory_block = np.random.randint(0, 256, size=(256, 256, 3), dtype=np.uint8)
raw_image = Image.fromarray(raw_memory_block)

print("INITIAL STATE (RAM)")
print(raw_memory_block[10:15, 10:15])
print(raw_image)
print("Data Type (raw memory block): ", type(raw_memory_block))
print("Data Type: ", type(raw_image))
print("Raw Shape (HWC): ", raw_memory_block.shape)

INITIAL STATE (RAM)
[[[ 82 153 114]
  [208  26 162]
  [126 178  12]
  [ 92 136  99]
  [ 26 205  47]]

 [[165 239  82]
  [119 245  80]
  [152 102 224]
  [ 17 161  32]
  [175   9 131]]

 [[226 250  96]
  [ 76 143 220]
  [112 101 111]
  [ 12 100 176]
  [138  57 214]]

 [[216  36 243]
  [123  35  33]
  [101 104 207]
  [ 89  77  87]
  [125  35  45]]

 [[228 110 168]
  [142 129  30]
  [ 80 121  26]
  [200 201 240]
  [ 90 244 103]]]
<PIL.Image.Image image mode=RGB size=256x256 at 0x79982674CD10>
Data Type (raw memory block):  <class 'numpy.ndarray'>
Data Type:  <class 'PIL.Image.Image'>
Raw Shape (HWC):  (256, 256, 3)


#### Image Transformation Pipeline Breakdown

The image pipeline sequentially converts a **raw PIL Image or NumPy array** into a **normalized PyTorch tensor** with a reshaped memory layout, making it ready for deep learning models.

##### 1. Input Data Format
* **Format**: A raw RGB image loaded via a library like PIL or NumPy.
* **Shape**: $H \times W \times C$ (Height, Width, Channels).
* **Data Range**: Integer values from $0$ to $255$.

##### 2. Step-by-Step Execution

###### 1. Convert to Tensor
* **Action**: `transforms.ToTensor()` processes the raw image.
* **Layout Shift**: Rearranges the dimensions from $H \times W \times C$ to $C \times H \times W$ (Channels, Height, Width) to match PyTorch expectations.
* **Value Scaling**: Scales all pixel values down from integers to a floating-point range of $[0.0, 1.0]$.

###### 2. Standardize Values
* **Action**: `transforms.Normalize()` applies a statistical shift to each color channel independently.
* **Formula**: It computes the new value for each pixel using the formula:
$$\text{Output} = \frac{\text{Pixel Value} - \text{Mean}}{\text{Std}}$$
* **Data Shift**: It shifts the pixel values from $[0.0, 1.0]$ to a range centered around zero (typically between roughly $-2.1$ and $2.6$), matching the distribution of the ImageNet dataset.

###### 3. Output Data Format
* **Format**: A `torch.Tensor` object ready for GPU processing.
* **Shape**: $C \times H \times W$ ($3$ channels $\times$ Height $\times$ Width).
* **Data Range**: Normalized floating-point numbers centered around $0.0$.

The raw image passes through structural rearrangement and mathematical scaling to produce an optimized, model-ready tensor.


In [ ]:
# Image transformation pipeline

# Standard deep learning vision pipeline
# Declaraing mathematical opeations the CPU will perform on the image before sending it to the GPU
# Input data format: Memory Layout (Shape) = HWC, Format = RGB, Data Range = 0 to 255 per pixel per layer.
image_pipeline = transforms.Compose([
    # First operation: trasnforms.ToTensor()
    # Convert the image into tensor
    # Changes the memory layout from HWC (256, 256, 3) to CHW (3, 256, 256)
    # Casts the uint8 integers to float32 and divides every pixel by 255.0
    transforms.ToTensor(),
    # Second operation: transforms.Normalize()
    # Execute the statistical shift
    # Using the mean and standard deviation numbers from ImageNet dataset (universal standard)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],    # Mean for R, G, B
                         std=[0.299, 0.244, 0.255]      # Standard Deviation for R, G, B
    )
])
# Output data format: torch.Tensor() ready for GPU, Memory Layout (Shape) = CHW, Data Range = Normalized floating-point numbers centered around 0.0

In [ ]:
# Exuting the image transformation pipeline and loading it onto the GPU

print("Executing Image Transformation Pipeline")

# Run the image through the image transform pipeline
processed_tensor = image_pipeline(raw_image)
print("Updated Data Type:", processed_tensor.dtype)
print("Updated Shape:", processed_tensor.shape)
print(f"Min Max Range of data: Min={processed_tensor.min():.2f}, Max={processed_tensor.max():.2f}")

# Construct image batch (video) for GPU loading
batch_tensor = processed_tensor.unsqueeze(0)
print(f"Batch Shape (B, C, H, W):", batch_tensor.shape)

# Load the batch to the GPU through PCIE
gpu_batch = batch_tensor.to(device)
print("Final State (VRAM)")
print("Device:", gpu_batch.device)

# The image is now in GPU VRAM and ready to servce for CNN training

Executing Image Transformation Pipeline
Updated Data Type: torch.float32
Updated Shape: torch.Size([3, 256, 256])
Min Max Range of data: Min=-1.87, Max=2.33
Batch Shape (B, C, H, W): torch.Size([1, 3, 256, 256])
Final State (VRAM)
Device: cuda:0


### Convolution

#### Convolution Operation (Edge Detection) in Image

**1. The Input Tensor (The Image)**
We load a small $6 \times 6$ pixel region of the road into RAM.
The left half is bright white (pixel value 10), and the right half is dark asphalt (pixel value 0).
<br><br>
$$
\text{Input } (I) =
\begin{bmatrix}
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0 \\
10 & 10 & 10 & 0 & 0 & 0
\end{bmatrix}
$$
<br>
**2. The Learnable Weights (The Kernel)**
Instead of random numbers, let's assume our CNN has already "learned" the exact weights required to detect a vertical edge. This specific $3 \times 3$ matrix is known mathematically as the Prewitt vertical operator.
<br><br>
$$
\text{Kernel } (K) =
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
**3. The Execution: Scanning a Flat Region**
The GPU places the $3 \times 3$ kernel over the top-left corner of the input image. This region is entirely bright white `(10)`.
The CPU/GPU computes the element-wise multiplication and sums the result (the dot product):
<br><br>
$$
\text{Patch} =
\begin{bmatrix}
10 & 10 & 10 \\
10 & 10 & 10 \\
10 & 10 & 10
\end{bmatrix}
*
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
$$
\text{Calculation: } (10\times1 + 10\times0 + 10\times-1) \times 3 \text{ rows}
$$
<br>
$$
\text{Result: } (10 + 0 - 10) + (10 + 0 - 10) + (10 + 0 - 10) = \mathbf{0}
$$
<br>
*CS Conclusion:* The output is 0. The kernel mathematically proves there is no vertical edge in this specific patch of memory.

**4. The Execution: Hitting the Edge**
The GPU strides the kernel over to the middle of the image, where the 10s meet the 0s.
<br><br>
$$
\text{Patch} =
\begin{bmatrix}
10 & 10 & 0 \\
10 & 10 & 0 \\
10 & 10 & 0
\end{bmatrix}
*
\begin{bmatrix}
 1 & 0 & -1 \\
 1 & 0 & -1 \\
 1 & 0 & -1
\end{bmatrix}
$$
<br>
$$
\text{Calculation: } (10\times1 + 10\times0 + 0\times-1) \times 3 \text{ rows}
$$
<br>
$$
\text{Result: } (10 + 0 + 0) + (10 + 0 + 0) + (10 + 0 + 0) = \mathbf{30}
$$
<br>
*CS Conclusion:* The output is a massive positive number (30). The kernel has successfully "activated." This high numerical value tells the next layer of the neural network: *"I found a stark vertical boundary at this exact spatial coordinate."*

**5. The Output Feature Map**
After sliding across the entire $6 \times 6$ image (with a stride of 1 and no padding), the resulting Feature Map is a $4 \times 4$ tensor.

$$
\text{Feature Map } =
\begin{bmatrix}
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0 \\
0 & 30 & 30 & 0
\end{bmatrix}
$$
<br>
The network has successfully transformed a grid of raw pixel intensities into a mathematical map of geometric features. If we configure a convolutional layer with 64 `out_channels`, the GPU runs 64 of these distinct filters simultaneously, extracting horizontal edges, diagonal lines, and color blobs all in one massive parallel operation.

In [2]:
# SIMULATING THE CONVOLUTION OPERATION

import torch
import torch.nn as nn

In [5]:
# Harware setup, sample data allocation then moving the sample data to GPU

# Set the device as GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Allocate a sample batch of images in VRAM
# Batch Size (No. of images) (B) = 32, RGB Channels (C) = 3, Height (H) = 64, Width (W) = 64
batch_size = 32
input_tensor = torch.randn(size=(batch_size, 3, 64, 64)).to(device)
print(f"Input tensor shape: {input_tensor.shape}")
print(f"Input tensor device: {input_tensor.device}")

cuda
Input tensor shape: torch.Size([32, 3, 64, 64])
Input tensor device: cuda:0
